# Auditing a model on `bank`

A full isitfair audit of a logistic regression fitted to the public
`bank` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py bank

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "bank"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
bank: outcome y == 'yes' (term deposit subscribed); threshold 0.2; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 13564, prevalence = 0.117
age_group: 5 levels, 0 missing
job: 12 levels, 0 missing
marital: 3 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

calibration n = 6782, audit n = 6782


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=6782, events=793, threshold=0.2, n_bootstrap=50, random_state=42, attributes: age_group (5), job (12), marital (3), age_group × job (58))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,age_group,Overall,6782.0,793.0,0.749653,0.721483,0.762689
1,age_group,18-29,747.0,125.0,0.785608,0.734568,0.821088
2,age_group,30-39,2693.0,272.0,0.728717,0.701480,0.759906
3,age_group,40-49,1712.0,157.0,0.709685,0.648316,0.741978
4,age_group,50-59,1358.0,144.0,0.730911,0.689853,0.772644
...,...,...,...,...,...,...,...
89,age_group × job,60+ / self-employed,5.0,2.0,0.500000,NaN,NaN
90,age_group × job,60+ / services,4.0,1.0,0.000000,NaN,NaN
91,age_group × job,60+ / technician,12.0,3.0,0.518519,NaN,NaN
92,age_group × job,60+ / unemployed,6.0,4.0,0.500000,NaN,NaN


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,age_group,Overall,6782.0,0.009466,1.033763,0.086231,0.010016,0.009307
1,age_group,18-29,747.0,0.198120,1.063079,0.113279,0.033988,0.025179
2,age_group,30-39,2693.0,-0.126326,0.990013,0.078058,0.014403,0.011805
3,age_group,40-49,1712.0,-0.047844,1.077855,0.071495,0.016559,0.014102
4,age_group,50-59,1358.0,0.007334,1.120574,0.080847,0.021983,0.016082
...,...,...,...,...,...,...,...,...
83,age_group × job,60+ / self-employed,5.0,1.786239,-0.040299,0.348691,0.447801,0.447801
84,age_group × job,60+ / services,4.0,1.021216,-54.851301,0.237867,0.333902,0.333902
85,age_group × job,60+ / technician,12.0,0.459499,0.149979,0.206526,0.323515,0.169418
86,age_group × job,60+ / unemployed,6.0,1.743538,0.141721,0.353661,0.541227,0.515958


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (8200, 7) | ROC curves: (8200, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
120,age_group,0.267959,0.21986,0.321465
381,job,0.229248,NaN,NaN
462,marital,0.036456,0.01766,0.056453
1643,age_group × job,0.740741,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

547 of 930 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,age_group,Overall,6782.0,10.105354,10.0,0.431299,10.0
1,age_group,18-29,747.0,13.985040,10.0,0.173675,10.0
2,age_group,30-39,2693.0,10.371236,10.0,0.408549,10.0
3,age_group,40-49,1712.0,10.061745,10.0,0.435093,10.0
4,age_group,50-59,1358.0,11.755688,10.0,0.301740,10.0
...,...,...,...,...,...,...,...
80,age_group × job,60+ / self-employed,5.0,21.200119,5.0,0.000742,5.0
81,age_group × job,60+ / services,4.0,18.187607,4.0,0.001134,4.0
82,age_group × job,60+ / technician,12.0,15.344227,10.0,0.120007,10.0
83,age_group × job,60+ / unemployed,6.0,16.057227,6.0,0.013450,6.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,age_group × job,18-29 / admin.,112,14,True,0.167336,0.125000,0.500761
1,age_group × job,18-29 / blue-collar,146,16,True,0.167336,0.109589,0.434857
2,age_group × job,18-29 / entrepreneur,15,2,True,0.167336,0.133333,0.882207
3,age_group × job,18-29 / housemaid,8,3,True,0.167336,0.375000,0.933523
4,age_group × job,18-29 / management,112,25,True,0.167336,0.223214,0.500761
5,age_group × job,18-29 / retired,1,0,True,0.167336,0.000000,0.991177
6,age_group × job,18-29 / self-employed,29,8,True,0.167336,0.275862,0.794823
7,age_group × job,18-29 / services,78,4,True,0.167336,0.051282,0.590211
8,age_group × job,18-29 / student,103,28,True,0.167336,0.271845,0.521690
9,age_group × job,18-29 / technician,121,22,True,0.167336,0.181818,0.481447


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by age_group


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,age_group,Overall,6782.0,793.0,0.009466,-0.071228,0.086328,1.033763,0.966390,...,0.081337,0.090000,0.010016,0.009307,0.030743,NaN,NaN,25.0,False,False
1,none,age_group,18-29,747.0,125.0,0.198120,-0.022935,0.370238,1.063079,0.874618,...,0.099746,0.128547,0.033988,0.025179,0.061245,NaN,NaN,25.0,False,False
2,none,age_group,30-39,2693.0,272.0,-0.126326,-0.266456,-0.020128,0.990013,0.861389,...,0.070177,0.084853,0.014403,0.011805,0.020795,NaN,NaN,25.0,False,False
3,none,age_group,40-49,1712.0,157.0,-0.047844,-0.169321,0.063726,1.077855,0.849158,...,0.064253,0.078830,0.016559,0.014102,0.020882,NaN,NaN,25.0,False,False
4,none,age_group,50-59,1358.0,144.0,0.007334,-0.195334,0.211057,1.120574,0.963737,...,0.069025,0.093127,0.021983,0.016082,0.021723,NaN,NaN,25.0,False,False
5,none,age_group,60+,272.0,95.0,0.538648,0.283723,0.844409,0.587290,NaN,...,0.191304,0.249345,0.115005,0.100552,0.152574,NaN,NaN,25.0,False,True
6,none,age_group,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,0.664974,NaN,NaN,NaN
7,common,age_group,Overall,6782.0,793.0,-0.001349,-0.083611,0.081149,0.932067,0.836471,...,0.081098,0.089769,0.007906,0.004949,0.032734,NaN,NaN,25.0,False,False
8,common,age_group,18-29,747.0,125.0,0.166162,-0.049672,0.340399,0.960879,0.778942,...,0.099328,0.128296,0.024784,0.018845,0.061245,NaN,NaN,25.0,False,False
9,common,age_group,30-39,2693.0,272.0,-0.143107,-0.292747,-0.034434,0.847527,0.666313,...,0.070239,0.085163,0.018042,0.016720,0.021816,NaN,NaN,25.0,False,False


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, age_group
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.009466           1.033763 0.086231 0.010016
   18-29               0.198120           1.063079 0.113279 0.033988
   30-39              -0.126326           0.990013 0.078058 0.014403
   40-49              -0.047844           1.077855 0.071495 0.016559
   50-59               0.007334           1.120574 0.080847 0.021983
     60+               0.538648           0.587290 0.212510 0.115005
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, age_group
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.024577           0.711248 0.086202 0.015942
   18-29               0.036912           1.141860 0.112975 0.041195
   30-39              -0.114408           0.740188 0.078888 0.017004
   40-49               0.140753       

In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
6,equal_opportunity_difference,0.179683,0.131491,0.255529
7,predictive_equality,0.056134,0.036988,0.082529
8,equalized_odds_gap,0.179683,0.131491,0.255529
9,statistical_parity,0.020607,0.015870,0.053842


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

isitfair\report.py:296: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


isitfair\report.py:211: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


isitfair\report.py:330: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


wrote extra_examples\reports\bank.html
